# Temperatura: escopo, evidências e próximas etapas

Revisão de 03/11/2026 do áudio de Fabrício, de `instrucoes.md` e dos resultados
em `refinamento/`. **O desenvolvimento está dentro do escopo; o produto nacional
validado ainda não está pronto.** Este notebook lê apenas resultados agregados
e não acessa séries por estação nem o Earth Engine.

O relatório detalhado está em [docs/auditoria_escopo.md](../docs/auditoria_escopo.md).


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if (ROOT / 'refinamento').is_dir():
    ROOT = ROOT / 'refinamento'
elif ROOT.name == 'notebooks':
    ROOT = ROOT.parent
TABLES = ROOT / 'resultados' / 'tabelas'
if not TABLES.is_dir():
    raise RuntimeError('Abra este notebook a partir de refinamento ou de refinamento/notebooks.')
print('Tabelas agregadas encontradas.')


Tabelas agregadas encontradas.


## O que já existia e foi conferido

- Dados reais de Tmax/Tmin, normais mensais 1991–2020 e controle de qualidade inicial.
- 120 estações elegíveis: 102 no desenvolvimento e 18 reservadas, sem avaliação.
- Comparações espaciais de regressão linear, random forest e LightGBM.
- 39 GeoTIFFs de uma prévia de Curitiba: 12 meses + anual, cada qual com temperatura,
  dispersão de modelos e flag de extrapolação. Os arquivos têm pixels de 30 m.
- A auditoria verificou os 39 rasters, os hashes e a coerência entre mensal/anual.

Resolução de 30 m **não demonstra habilidade preditiva a 30 m**. A dispersão dos
modelos não é um intervalo preditivo calibrado. A flag min/max não é uma área
formal de aplicabilidade multivariada.


## C1: as quatro bases no período do projeto

Foram extraídas séries de Xavier, ERA5-Land e TerraClimate nas 102 estações de
desenvolvimento. A amostra comum tem 98 estações: quatro não têm pixels válidos
em todas as grades. As predições dos modelos foram reaproveitadas, conservando
seus treinos e dobras originais. As 18 estações reservadas continuam fechadas.


In [2]:
coverage = pd.read_csv(TABLES / 'c1_cobertura_1991_2020.csv')
print(coverage.query("mode == 'full_1991_2020'").to_string(index=False))
scores = pd.read_csv(TABLES / 'c1_modelos_2graus.csv')
table = (scores[scores.scope.isin(['global', 'annual'])]
         .groupby(['method', 'scope']).rmse.mean().unstack('scope')
         .rename(columns={'global': 'RMSE_mensal_C', 'annual': 'RMSE_anual_C'})
         .sort_values('RMSE_mensal_C'))
print('\nRMSE médio das 3 repetições; blocos de 2 graus:')
print(table.round(3).to_string())


          mode       source  expected_normals  valid_normals  complete_stations
full_1991_2020       xavier              1224           1212                101
full_1991_2020         era5              1224           1176                 98
full_1991_2020 terraclimate              1224           1224                102
full_1991_2020       chelsa              1224           1224                102

RMSE médio das 3 repetições; blocos de 2 graus:
scope                 RMSE_anual_C  RMSE_mensal_C
method                                           
M0_xavier                    0.236          0.241
M5_lightgbm                  0.456          0.526
M4_random_forest             0.460          0.531
M2_linear                    0.524          0.593
M0_chelsa                    0.722          0.798
M1_lapse_fixo                0.759          0.832
MX_mensal_referencia         0.753          0.836
M0_era5                      1.138          1.228
M0_terraclimate              1.227          1.267


![Comparação das bases](../resultados/figuras/c1_quatro_bases.png)

O LightGBM melhora o CHELSA, mas o critério de superar a grade Xavier **não foi
cumprido**. O Xavier pode ter usado as mesmas estações; a comparação não é
independente. O alvo observado usa 24–30 anos por mês, enquanto a normal integral
da grade usa os 30 anos. Não escolher a base final só por esse ranking.


In [3]:
paired = pd.read_csv(TABLES / 'c1_diferencas_2graus.csv')
print(paired.query("method == 'M5_lightgbm'")[
    ['reference', 'delta_rmse_c', 'ci025', 'ci975', 'n_blocks']].round(4).to_string(index=False))
print('\nDiferença = RMSE do LightGBM menos RMSE da referência; negativo favorece LightGBM.')
print('Intervalos por reamostragem de blocos, condicionais às predições de CV.')
print('\nC9 parcial: altitude, mesmas 98 estações:')
print(scores.query("scope == 'elevation' and method in ['M0_chelsa','M0_xavier','M5_lightgbm']")
      .groupby(['group','method'])[['n_stations','rmse']].mean().round(3).to_string())


      reference  delta_rmse_c   ci025   ci975  n_blocks
      M0_chelsa       -0.2724 -0.3402 -0.1946        69
      M0_xavier        0.2843  0.2203  0.3530        69
        M0_era5       -0.7022 -0.8195 -0.5829        69
M0_terraclimate       -0.7414 -0.9039 -0.5825        69

Diferença = RMSE do LightGBM menos RMSE da referência; negativo favorece LightGBM.
Intervalos por reamostragem de blocos, condicionais às predições de CV.

C9 parcial: altitude, mesmas 98 estações:
                           n_stations   rmse
group         method                        
300_to_800m   M0_chelsa          31.0  0.731
              M0_xavier          31.0  0.283
              M5_lightgbm        31.0  0.458
at_least_800m M0_chelsa          19.0  0.853
              M0_xavier          19.0  0.272
              M5_lightgbm        19.0  0.592
below_300m    M0_chelsa          48.0  0.817
              M0_xavier          48.0  0.195
              M5_lightgbm        48.0  0.538


## C12 parcial: retirar anos conhecidos

250 séries estação-mês, em 43 estações, têm 30 anos elegíveis e referências
completas. Retiramos anos e comparamos a normal estimada com a normal conhecida.
Todos os métodos recebem as mesmas máscaras; valores observados retirados não
entram no ajuste. São falhas de anos, não falhas diárias.


In [4]:
gaps = pd.read_csv(TABLES / 'c12_falhas_artificiais.csv')
print(gaps.query("source == 'era5' and scenario in ['random_24','recent_24','first_10','recent_10']")[
    ['scenario','method','n_stations','n_station_months','rmse','bias']].round(4).to_string(index=False))


 scenario          method  n_stations  n_station_months   rmse    bias
 first_10 available_years          43               250 0.3644 -0.2814
 first_10      difference          43               250 0.1997 -0.0243
 first_10 regression_fill          43               250 0.2255 -0.0686
random_24 available_years          43               250 0.0823  0.0012
random_24      difference          43               250 0.0397 -0.0003
random_24 regression_fill          43               250 0.0366 -0.0004
recent_10 available_years          43               250 0.4075  0.3086
recent_10      difference          43               250 0.2097  0.0159
recent_10 regression_fill          43               250 0.2084  0.0687
recent_24 available_years          43               250 0.1165  0.0855
recent_24      difference          43               250 0.0626  0.0102
recent_24 regression_fill          43               250 0.0635  0.0239


![Falhas artificiais](../resultados/figuras/c12_falhas_artificiais.png)

Em 2011–2020, a média disponível tem viés de +0,309 °C; o método das diferenças
com ERA5 reduz a +0,016 °C. O ajuste temporal merece continuar, mas esse ensaio
em séries longas não valida automaticamente as estações curtas. Os meses de
origem passam pelo QC de pelo menos 80% dos dias; não são séries diárias completas.


## C13: outra década e outras estações

68 estações, meses com pelo menos oito anos em ambas as décadas. O modelo
aprende observado menos ERA5 em 2001–2010 e é avaliado em 2011–2020 em blocos
espaciais excluídos do treino. O controle contemporâneo também usa outras
estações, mas aprende em 2011–2020. A referência ERA5 da década avaliada é
conhecida: é uma avaliação retrospectiva, não previsão meteorológica futura.


In [5]:
temporal = pd.read_csv(TABLES / 'c13_modelos_2graus.csv')
print(temporal.query("scope == 'global'").groupby('method')[['rmse','mae','bias']]
      .mean().round(4).to_string())
ci = pd.read_csv(TABLES / 'c13_diferencas_2graus.csv')
print('\nLightGBM treinado na década anterior:')
print(ci.query("method == 'M5_lightgbm_from_2001_2010'")[
    ['reference','delta_rmse_c','ci025','ci975']].round(4).to_string(index=False))


                                   rmse     mae    bias
method                                                 
M0_era5                          1.2668  1.1171 -0.9905
M2_linear_from_2001_2010         0.8490  0.6172 -0.0291
M2_linear_from_2011_2020         0.8396  0.6088  0.0183
M4_random_forest_from_2001_2010  0.7704  0.5542 -0.0319
M4_random_forest_from_2011_2020  0.7977  0.5749  0.0073
M5_lightgbm_from_2001_2010       0.7498  0.5420 -0.0019
M5_lightgbm_from_2011_2020       0.7465  0.5522  0.0439

LightGBM treinado na década anterior:
                 reference  delta_rmse_c   ci025   ci975
                   M0_era5        -0.517 -0.6943 -0.3299
M5_lightgbm_from_2011_2020         0.003 -0.0476  0.0456


A correção transferida melhora o ERA5, sem diferença conclusiva contra o
controle contemporâneo neste conjunto. Isso não prova transferência da correção
de CHELSA, validade retrospectiva a 1991–2000 ou habilidade a 30 m.

## Situação C1–C15


In [6]:
status = pd.read_csv(TABLES / 'status_comparacoes.csv').fillna('')
print(status[['comparacao','status','decisao_pendente']].to_string(index=False))


comparacao                       status                                                                                                                 decisao_pendente
        C1 executada_no_desenvolvimento                                                          Resolver definicao termica e independencia antes de escolher base final
        C2 executada_temperatura_CHELSA Gradientes global mensal e mensal regional ajustados no treino sem ganho conclusivo; fixo piorou; revisar escala do relevo em C5
        C3                      parcial                                                       RF e LightGBM sem vencedor definitivo; avancar na escada apos revisar alvo
        C4       executada_exploratoria                                                      Correcao promissora; extrapolacao e periodo ainda precisam ser demonstrados
        C5                     pendente                                                                                 Comparar relevo a 30/90/1000 m nas 

## Antes da entrega final

1. C11: comparar definições de temperatura usando dados horários reais e revisar QC.
2. Concluir C12 para falhas diárias e séries curtas, congelando alvo e partições.
3. C14/C15: conferir estações independentes das grades e a reprodução do Xavier.
4. C5–C8: escalas do relevo, grupos de covariáveis, sazonalidade e densidade de estações.
5. C9/C10: regiões/biomas, artefatos e região-piloto abrangente; depois produção nacional,
   teste reservado, incerteza/aplicabilidade e upload manual.

### Reproduzir os ensaios

Na pasta `refinamento/`, usando o ambiente já configurado:

```powershell
.venv/Scripts/python.exe -B -m codigo.bases_gee
.venv/Scripts/python.exe -B -m codigo.comparar_bases
.venv/Scripts/python.exe -B -m codigo.falhas_artificiais
.venv/Scripts/python.exe -B -m codigo.transferencia_temporal
.venv/Scripts/python.exe -B -m codigo.figuras_auditoria
```

Apenas a primeira linha consulta o GEE. Os resultados por estação permanecem em
diretórios ignorados. Fontes, conversões, limitações e referências estão no
[relatório](../docs/auditoria_escopo.md); decisões no [registro](../docs/registro.md).


## Atualização nacional — 07/11/2026

C7/C8 executadas para temperatura: estratégias de sazonalidade e frações de estações apenas no treino. Teste reservado preservado.
O documento [entrega_brasil.md](../docs/entrega_brasil.md) detalha requisitos, evidências e pendências.


In [7]:
season = pd.read_csv(TABLES / 'c7_c8_metricas_2graus.csv')
print(season.query("scope == 'global'").groupby('method').rmse.mean().to_string())
import json
progress = json.loads((TABLES / 'brasil_auditoria_producao.json').read_text(encoding='utf-8'))
print(json.dumps(progress, indent=2, ensure_ascii=False))


method
M0_chelsa                         0.794394
M2_linear_categorical_100pct      0.585149
M2_linear_harmonics_025pct        0.674905
M2_linear_harmonics_050pct        0.600677
M2_linear_harmonics_075pct        0.586930
M2_linear_harmonics_100pct        0.587761
M2_linear_none_100pct             0.615299
M2_linear_per_month_100pct        0.588260
M5_lightgbm_categorical_100pct    0.544291
M5_lightgbm_harmonics_025pct      0.620407
M5_lightgbm_harmonics_050pct      0.572374
M5_lightgbm_harmonics_075pct      0.544735
M5_lightgbm_harmonics_100pct      0.523906
M5_lightgbm_none_100pct           0.571579
M5_lightgbm_per_month_100pct      0.545605
{
  "planned_tiles": 743,
  "complete_temperature_tiles": 0,
  "partial_temperature_tiles": 1,
  "tiles_with_missing_covariates": 0,
  "processed_windows": 2,
  "valid_temperature_pixels": 524288,
  "output_bytes": 12371595,
  "national_temperature_complete": false,
  "rainfall_maps_produced": false,
  "scientifically_validated_national_product":

![Sazonalidade](../resultados/figuras/c7_sazonalidade.png)

![Densidade](../resultados/figuras/c8_densidade.png)

Os resultados são de desenvolvimento. Grade planejada e janelas parciais não equivalem à entrega nacional validada.


## C2 e preparação da chuva — 07/11/2026

Gradientes ajustados somente no treino, preservando as dobras e o teste reservado. O ganho mensal/regional não foi conclusivo.
A preparação da chuva gerou normais observadas; modelagem e mapas continuam pendentes.


In [8]:
gradients = pd.read_csv(TABLES / 'c2_gradientes_2graus.csv')
print(gradients.query("scope == 'global'").groupby('method').rmse.mean().to_string())
print(pd.read_csv(TABLES / 'chuva_sensibilidade_cobertura.csv').to_string(index=False))


method
M0_chelsa             0.794394
M1_fixo               0.825530
M1_global             0.788313
M1_mensal             0.788758
M1_mensal_regional    0.775111
 minimum_years  coverage_fraction  stations_12_months  eligible_station_months
            21                0.7                2971                    39596
            24                0.8                2360                    35244
            27                0.9                1159                    27149
